# Practice Exercises: SQL Subqueries and SQL with Pandas

**Level:** Beginner to intermediate  
**Tools:** Python, pandas, SQLite (`sqlite3`)

## Learning outcomes
By the end of these exercises, you should be able to:
- Explain what a SQL subquery is and when to use one.
- Use subqueries with `WHERE`, `IN`, and aggregate functions.
- Run SQL queries from Python using `sqlite3`.
- Load SQL query results into pandas DataFrames with `pd.read_sql_query()`.
- Use pandas to inspect, filter, group, and summarize SQL query results.

## Quick notes

### 1. SQL subqueries
A **subquery** is a query nested inside another SQL query. It can return a single value, a list of values, or a table-like result.

Example:
```sql
SELECT product_name, price
FROM products
WHERE price > (SELECT AVG(price) FROM products);
```
The inner query calculates the average product price. The outer query returns products priced above that average.

Common patterns:
- `WHERE price > (SELECT ...)` — compare against a value.
- `WHERE customer_id IN (SELECT ...)` — match values returned by another query.
- A subquery can be used in `FROM` to create a temporary result set.

### 2. SQL with pandas
Pandas can send SQL queries to a database and return the results as a DataFrame:
```python
df = pd.read_sql_query("SELECT * FROM customers", conn)
```
You can then use pandas methods such as `.head()`, `.shape`, `.groupby()`, and filtering to explore the results.

**Important:** Run the setup cell first. It creates a temporary SQLite database and sample tables for all exercises.

In [114]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Sample data
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "customer_name": ["Amina", "Brian", "Carol", "David", "Esther", "Farah"],
    "city": ["Nairobi", "Mombasa", "Nairobi", "Kisumu", "Nakuru", "Mombasa"]
})

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105],
    "product_name": ["Laptop", "Mouse", "Keyboard", "Monitor", "USB Drive"],
    "category": ["Electronics", "Accessories", "Accessories", "Electronics", "Accessories"],
    "price": [75000, 1200, 2500, 22000, 800]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009],
    "customer_id": [1, 2, 1, 3, 4, 5, 2, 6, 3],
    "product_id": [101, 102, 103, 104, 102, 105, 103, 101, 105],
    "quantity": [1, 2, 1, 1, 3, 4, 2, 1, 2],
    "order_date": ["2026-01-05", "2026-01-08", "2026-01-12", "2026-02-01",
                   "2026-02-04", "2026-02-15", "2026-03-02", "2026-03-10", "2026-03-12"]
})

customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")
orders.to_sql("orders", conn, index=False, if_exists="replace")

print("Database created. Tables: customers, products, orders")


Database created. Tables: customers, products, orders


## Part A — SQL Subqueries

Use SQL in `pd.read_sql_query()` or `conn.execute()` to answer each question. Replace the `YOUR QUERY HERE` placeholders.

### Exercise 1: Products above the average price
Return `product_name` and `price` for products whose price is greater than the average price of all products.

In [115]:
query = '''
SELECT product_name, price
FROM products
GROUP BY product_name, price
HAVING price > (SELECT AVG(price) FROM products);
'''
pd.read_sql_query(query, conn)

,product_name,price
0,Laptop,75000
1,Monitor,22000


### Exercise 2: Customers who have placed an order
Use a subquery with `IN` to return the names of customers who have placed at least one order. Return `customer_name` and `city`.

In [116]:
query = '''
SELECT customer_name, city
FROM customers
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING COUNT(customer_id) >= 1
);
'''
pd.read_sql_query(query, conn)

,customer_name,city
0,Amina,Nairobi
1,Brian,Mombasa
2,Carol,Nairobi
3,David,Kisumu
4,Esther,Nakuru
5,Farah,Mombasa


### Exercise 3: Products that have never been ordered
Use a subquery with `NOT IN` to find products that do not appear in the orders table. Return `product_id` and `product_name`.

In [117]:
query = '''
SELECT product_id, product_name
FROM products
WHERE product_id NOT IN (
    SELECT product_id
    FROM orders
    WHERE product_id is NOT NULL
);
'''
pd.read_sql_query(query, conn)

,product_id,product_name


### Exercise 4: Orders above the average order quantity
Return all columns from orders where `quantity` is greater than the average quantity across all orders.

In [118]:
query = '''
SELECT * 
FROM orders
WHERE quantity > (
    SELECT AVG(quantity)
    FROM orders
);

'''
pd.read_sql_query(query, conn)

,order_id,customer_id,product_id,quantity,order_date
0,1002,2,102,2,2026-01-08
1,1005,4,102,3,2026-02-04
2,1006,5,105,4,2026-02-15
3,1007,2,103,2,2026-03-02
4,1009,3,105,2,2026-03-12


### Exercise 5: Customers with more than one order
Use a subquery to identify customers whose IDs occur more than once in the orders table. Return each matching customer once, with their name.

In [119]:
query = '''
SELECT customer_name
FROM customers
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING COUNT (*)>1
);
'''
pd.read_sql_query(query, conn)

,customer_name
0,Amina
1,Brian
2,Carol


### Exercise 6: Products priced above their category average
For each product, compare its price with the average price of products in the same category. Return the product name, category, and price for products above their category average.

*Hint:* This is a **correlated subquery** because the inner query refers to the outer query.

In [120]:
query = '''
SELECT p.product_name, p.category, p.price
FROM products p
WHERE p.price >(
    SELECT AVG(price)
    FROM products
    WHERE category = p.category
);

'''
pd.read_sql_query(query, conn)

,product_name,category,price
0,Laptop,Electronics,75000
1,Keyboard,Accessories,2500


## Part B — SQL with Pandas

### Exercise 7: Load a table into pandas
Load the full `customers` table into a DataFrame named `customers_df`. Display the first five rows and the DataFrame shape.

In [121]:
# Write your code here
customers_df = pd.read_sql_query(
    """
    SELECT * 
    FROM customers
    """,conn
)
customers_df.head()


,customer_id,customer_name,city
0,1,Amina,Nairobi
1,2,Brian,Mombasa
2,3,Carol,Nairobi
3,4,David,Kisumu
4,5,Esther,Nakuru


In [122]:
customers_df.shape

(6, 3)

### Exercise 8: SQL query results as a DataFrame
Write a SQL query that returns each order's `order_id`, `customer_id`, `product_id`, and `quantity`. Load the result into `orders_df`, then display its first five rows.

In [123]:
# Write your code here
orders_df = pd.read_sql_query(
"""
SELECT order_id, customer_id, product_id, quantity
FROM orders
""",conn)
orders_df.head()


,order_id,customer_id,product_id,quantity
0,1001,1,101,1
1,1002,2,102,2
2,1003,1,103,1
3,1004,3,104,1
4,1005,4,102,3


### Exercise 9: Total quantity ordered per product
Use SQL to calculate total quantity ordered for each `product_id`. Load the result into a DataFrame called `product_totals`. Sort it from highest to lowest total quantity.

In [124]:
# Write your code here
product_totals_df = pd.read_sql_query(
"""
SELECT product_id, SUM(quantity) AS total_quantity
FROM orders
GROUP BY product_id
ORDER BY quantity DESC
""",conn
)
product_totals_df

,product_id,total_quantity
0,105,6
1,102,5
2,104,1
3,103,3
4,101,2


### Exercise 10: Merge SQL data with pandas
Load orders and products into separate DataFrames. Use `pd.merge()` to add each product's name and price to every order. Display `order_id`, `product_name`, `quantity`, and `price`.

In [125]:
# Write your code here
orders_df = pd.read_sql_query("SELECT * FROM orders",conn)
products_df = pd.read_sql_query("SELECT * FROM products",conn)
merged_df = pd.merge(orders_df,products_df, on="product_id")
merged_df[["order_id","product_name","quantity","price"]]

,order_id,product_name,quantity,price
0,1001,Laptop,1,75000
1,1002,Mouse,2,1200
2,1003,Keyboard,1,2500
3,1004,Monitor,1,22000
4,1005,Mouse,3,1200
5,1006,USB Drive,4,800
6,1007,Keyboard,2,2500
7,1008,Laptop,1,75000
8,1009,USB Drive,2,800


### Exercise 11: Calculate order value using pandas
Using the merged DataFrame from Exercise 10, create a new column called `order_value` using `quantity * price`. Display the five orders with the highest order value.

In [126]:
# Write your code here
merged_df["order_value"]= merged_df["quantity"] * merged_df["price"]
merged_df.sort_values("order_value",ascending=False).head()


,order_id,customer_id,product_id,quantity,order_date,product_name,category,price,order_value
0,1001,1,101,1,2026-01-05,Laptop,Electronics,75000,75000
7,1008,6,101,1,2026-03-10,Laptop,Electronics,75000,75000
3,1004,3,104,1,2026-02-01,Monitor,Electronics,22000,22000
6,1007,2,103,2,2026-03-02,Keyboard,Accessories,2500,5000
4,1005,4,102,3,2026-02-04,Mouse,Accessories,1200,3600


### Exercise 12: Revenue by customer city
Join customers, orders, and products using SQL. Calculate revenue as `quantity * price` and return total revenue by city, sorted from highest to lowest. Load the result into a pandas DataFrame named `city_revenue`.

In [127]:
# Write your code here
city_revenue_df = pd.read_sql_query(
"""
SELECT c.city, SUM(quantity * price) AS total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.city
ORDER BY total_revenue DESC
""",conn
)
city_revenue_df

,city,total_revenue
0,Nairobi,101100
1,Mombasa,82400
2,Kisumu,3600
3,Nakuru,3200


### Exercise 13: Filter and summarize in pandas
Load the results of a SQL query containing each order's date, customer, product name, quantity, and price into a DataFrame. In pandas:
1. Keep orders with quantity of at least 2.
2. Create an `order_value` column.
3. Group by `product_name` and calculate total quantity and total value.
4. Sort by total value descending.

In [128]:
# Write your code here
df = pd.read_sql_query(
"""
SELECT o.order_date, c.customer_name, p.product_name, o.quantity, p.price
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
""",conn
);
df = df[df["quantity"] >= 2].copy()


In [129]:
#create an order_value_column
df["order_value"] = df["quantity"] * df["price"]

In [130]:
#group by product_name and calculate total_quantity & total_value & sort the values in DESC
summary = df.groupby("product_name").agg(
    total_quantity = ("quantity","sum"),
    total_value = ("order_value","sum")
).sort_values("total_value", ascending=False)
summary

,total_quantity,total_value
product_name,,
Mouse,5,6000
Keyboard,2,5000
USB Drive,6,4800


### Exercise 14: Challenge — compare SQL and pandas
Calculate total revenue per product in **two ways**:
1. Using a SQL query and `pd.read_sql_query()`.
2. Loading the necessary tables into pandas, merging them, calculating revenue, and grouping by product.

Compare the results. Do they agree? Briefly explain why they should.

In [131]:
# Method 1: SQL
sql_revenue = pd.read_sql_query(
"""
SELECT p.product_name, SUM(o.quantity * p.price) AS total_revenue
FROM orders o 
JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_revenue DESC
""",conn
);
sql_revenue


,product_name,total_revenue
0,Laptop,150000
1,Monitor,22000
2,Keyboard,7500
3,Mouse,6000
4,USB Drive,4800


In [132]:

# Method 2: pandas
orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
products_df = pd.read_sql_query("SELECT * FROM products", conn)

m = pd.merge(orders_df, products_df, on="product_id")
m["revenue"] = m["quantity"] * m["price"]
pandas_revenue = (m.groupby("product_name", as_index=False)["revenue"]
                   .sum()
                   .sort_values("product_name")
                   .reset_index(drop=True))
pandas_revenue


,product_name,revenue
0,Keyboard,7500
1,Laptop,150000
2,Monitor,22000
3,Mouse,6000
4,USB Drive,4800


In [133]:

comparison = pd.merge(
    sql_revenue,
    pandas_revenue,
    on="product_name",
    suffixes=("_sql", "_pandas")
)

comparison["match"] = comparison["total_revenue"].round(2).eq(
    comparison["revenue"].round(2)
)

print(comparison)
print("Do the results agree?", comparison["match"].all())

conn.close()

  product_name  total_revenue  revenue  match
0       Laptop         150000   150000   True
1      Monitor          22000    22000   True
2     Keyboard           7500     7500   True
3        Mouse           6000     6000   True
4    USB Drive           4800     4800   True
Do the results agree? True


## Reflection questions
1. What is the difference between a subquery and a join?
2. What is a correlated subquery?
3. What does `pd.read_sql_query()` return?
4. Give one reason to do an operation in SQL and one reason to do it in pandas.
5. Why should you be careful when using `NOT IN` if a subquery might return `NULL`?

## Submission checklist
- [ ] I ran the setup cell successfully.
- [ ] I completed all 14 exercises.
- [ ] I checked that my SQL statements run without errors.
- [ ] I can explain my approach in my own words.
